<h1>Chapter 9 - Multimodal Understanding</h1>
<i>Analyzing Images with your Agent.</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 9 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Choosing Your LLM

At the beginning of every chapter, we start by choosing the LLM that we want to use:

In [1]:
import os
from illustrated_agents.chapters.ch5_native import LLM

# Ollama through OpenAI API
llm = LLM(model="gemma4:e4b", backend="openai", api_base="http://localhost:11434/v1/", think=True)

# Llama.cpp server
# llm = LLM(model="openai/gemma-4-E4B-it-Q4_K_M", backend="litellm", api_base="http://localhost:8080", think=True)

# LM Studio
# llm = LLM(model="lm_studio/gemma-4-E4B-it", backend="litellm", api_base="http://localhost:1234/v1", think=True)

# Google's Gemini / Gemma
# os.environ['GEMINI_API_KEY'] = "YOUR_API_KEY"
# llm = LLM(model="gemini/gemini-2.5-flash", backend="litellm", api_key=None)

## 2 - Adding Multimodal Understanding

The model that we have been using `Gemma 4 E4B` is a multimodal model and is capable of processing images, audio, and video alongside text. In `Ollama` this only requires parsing the `messages` that we have been leveraging in a special way, namely like so:

```json
[
    {
        "role": "user",
        "content": [
                        {
                            "type": "text",
                            "text": "What’s in this image?"
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                            "url": "https://www.oreilly.com/covers/urn:orm:book:9798341662681/300w/"
                            }
                        }
                    ]
    }
]
```

Note how the `content` key now how to separate dictionaries, one containing `text` and the other `image_url`. This allows `Ollama` to process the image and feed it to the LLM. This means that we will have to adjust how the message structure is being used, which requires two changes.

* `memory.py` - Add a parameter to add the `"image_url"`
* `agent.py` - Only add the `"image_url"` to memory when the user provides an image url

Let's explore these changes, starting with `memory.py`:

In [20]:
class Memory:
    """Simple memory module to store conversation history."""

    def __init__(self):
        self.messages = []

    def add(self, role: str, content: str, tool_call: dict = None, image_data: str = None):
        """Add a message to memory."""
        # Image
        if image_data:
            content = [
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_data}"}},
                {"type": "text", "text": content}, 
            ]
        # Main message
        message = {"role": role, "content": content}

        # Tool call
        if tool_call:
            message["tool_calls"] = [tool_call]

        # Append message to memory
        self.messages.append(message)

    def get_messages(self) -> list[dict]:
        """Get all messages."""
        return self.messages

Note how straightforward these changes are, we merely need to update `.add` with the `image_url`. The changes are minimal:

In [21]:
from illustrated_agents.chapters.ch9 import memory_diff; memory_diff

## 3 - Updating `TinyAgent`

The changes to the `TinyAgent` are also minimal and instead of showing you the full code for the `TinyAgent`, we are going to create a new class that inherits all capabilities but simply adds the `image_url=image_url` when the user's task is first added to `Memory`:

In [22]:
from illustrated_agents.chapters import ch6_skills


class TinyAgent(ch6_skills.TinyAgent):
    def run(self, task: str, image_data: str = None) -> str:
        """Run the agent on a task."""
        self.memory.add("user", task, image_data=image_data)

        # `Autonomy` loop
        for step in range(self.planner.max_steps):
            result = self._step()
            if result is not None:
                return result

        return "Max steps reached without completion."

Only two lines of code need to be changed in order to add this multimodal capabilities to your `TinyAgent`:

In [23]:
from illustrated_agents.chapters.ch9 import tinyagents_diff; tinyagents_diff

## 3 - Running the Multimodal Agent

Now that you have the necessary components, you can create your `TinyAgent` and give it an image to analyze.

In [28]:
from illustrated_agents.chapters.ch5_native import NativeTools, LLM
from illustrated_agents.chapters.ch6_native import NativeReAct
from illustrated_agents.chapters.ch6_skills import Skills

# Multimodal Agent
agent = TinyAgent(
    llm=llm, 
    tools=NativeTools(), 
    memory=Memory(),
    skills=Skills(),
    planner=NativeReAct()
)

Next up, we will download the cover of "An Illustrated Guide to AI Agents" and encode it to base64 so that the openai endpoint can properly process the image:

In [29]:
import base64
import httpx

# Download and encode the image
image_url = "https://www.oreilly.com/covers/urn:orm:book:9798341662681/300w/"
image_data = base64.b64encode(httpx.get(image_url).content).decode("utf-8")

Finally, we simply ask the Agent which animal is on the cover. If it can correctly view the image then it should get the answer correct:

In [30]:
# Query - We ask it to access an image
query = "Which animal is on the cover of 'An Illustrated Guide to AI Agents'?"

# Run the agent
agent.run(query, image_data=image_data)

"The animal on the cover of 'An Illustrated Guide to AI Agents' is a **dolphin**."

The answer is correct, let's see how the model got to that conclusion:

In [32]:
from rich import print as pprint

pprint(agent.memory.get_messages())

[
    {
        'role': 'system',
        'content': 'You are a helpful AI agent.\n\n\n\n\n# Skills\n\nYou have specialized skills available. To use 
a skill,\ncall it like a tool by referencing their name.\n\nThe skill will provide detailed instructions for 
completing the task.\n\nAvailable skills:\n\n'
    },
    {
        'role': 'user',
        'content': [
            {
                'type': 'image_url',
                'image_url': {
                    'url': 
'data:image/jpeg;base64,/9j/2wBDAAMCAgMCAgMDAwMEAwMEBQgFBQQEBQoHBwYIDAoMDAsKCwsNDhIQDQ4RDgsLEBYQERMUFRUVDA8XGBYUGBI
UFRT/2wBDAQMEBAUEBQkFBQkUDQsNFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBQUFBT/wAARCAGKASwDAREAAh
EBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0Kxw
RVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ip
qrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtRE
AAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVl
dYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09
fb3+Pn6/9oADAMBAAIRAxEAPwD9RrWwtjbRf6PF9xf4B6UAS/YLb/n3i/74H+FAB9gtv+feL/vgf4UAH2C2/wCfeL/vgf4UAH2C2/594v8Avgf4UAH2
C2/594v++B/hQAfYLb/n3i/74H+FAB9gtv8An3i/74H+FAB9gtv+feL/AL4H+FAB9gtv+feL/vgf4UAH2C2/594v++B/hQAfYLb/AJ94v++B/hQAfYL
b/n3i/wC+B/hQAfYLb/n3i/74H+FAB9gtv+feL/vgf4UAH2C2/wCfeL/vgf4UAH2C2/594v8Avgf4UAH2C2/594v++B/hQAfYLb/n3i/74H+FAB9gtv
8An3i/74H+FAB9gtv+feL/AL4H+FAB9gtv+feL/vgf4UAH2C2/594v++B/hQAfYLb/AJ94v++B/hQAfYLb/n3i/wC+B/hQAfYLb/n3i/74H+FAB9gtv
+feL/vgf4UAH2C2/wCfeL/vgf4UAH2C2/594v8Avgf4UAH2C2/594v++B/hQAfYLb/n3i/74H+FAB9gtv8An3i/74H+FAB9gtv+feL/AL4H+FAB9gtv
+feL/vgf4UAH2C2/594v++B/hQAfYLb/AJ94v++B/hQAfYLb/n3i/wC+B/hQBj6xZW4uVxBF9z+4PU0AbVp/x6xf7i/yoAmoAKACgAoAKACgAoAKACg
AoAKACgAoAKACgAoAKACgAoAKACgAoAKACgAoAKACgAoAKACgAoAKACgAoAKACgDF1n/j6X/c/qaANS0/49Yv9xf5UATUAct8QfiVofwz0u3vNZlnL3
c4tbOysrd7m6u5iCRHFEgLO2ATwMADJIFYVq8KCTn126t+iPWy7LMTmlR08Ol7qvJtqMYrvKTskv6Ry2kftG+FNYaOBINattSF2LO60y70qaG6sWMTy
q88bDKRsqNiTlSeM8HHPHG0paa37Wd16/5nq1+GsdQvJuDhbmUlOLjLVRai09Wm1eO67bFXwl+054R8XXOmRRWniDTYtVhabTrvVdFuLW3vgsRl2wys
ux2MaswAPIHGaVPHUqjSSavtdNJ9dGbY3hXH4KM5SlCTg0pKM4ylG75feindK7s9NHuM0H9p3wz4g0a51mHR/FVvosGmTaudUu9AuIbV7eOMyFklYbW
yo+UA/N2pQx1OcXNRla172drepWI4VxmGqxw8qlJ1HNQ5VUi5czdrOKd1Z79up3XhH4g6P42u9WttMkleXS5IIrkSRFAGlgjnTGevySofYkjtXVTrQq
tqPS34q58/jMuxGBjTnXVlNNrW/wAMnF/imYY+OXhb/hFtE8SPPcxaJqupNpUV7JbsI4ZxM8A80/8ALNTJGVDnjJXpkVl9apckanRu1/nbX5ne8hxv1
mrhEk6lOPO1fVrlUvd7tRd7LW1+xf0z4raJrWuanpmnreXrabqKaTc3MFszQLclNzRiToSgwHx90kA85xccRCUnGOtnb5/1uc1XKcTQo061W0eeLmk2
r8t7J28/s91qdlXSeMFABQAUAFABQAUAFABQAUAFABQAUAFABQAUAFABQAUAFABQAUAFABQAUAFABQAUAYus/wDH0v8Auf1NAGpaf8esX+4v8qAJqAP
KPjF4Y8Qr4u8F+OPDelJ4ju/DhvIZ9FNwsElxBcxorPC7/IJUMa4DEBlZhkZrz8TTqc8K1NXcb6eT7eZ9dkuLwn1XE5bjKns41uRqdm0pQbaUkteV3e
17NJ2ZjeGfC/irxj438VeOtb8PN4VNz4f/ALA07RprmKa5mUO8pmnMZKKdzBVUM2BuJIzis6dOrUqTrTjy6WS697v9DtxWLwWCwdDLMNW9ry1PaSmk1
FOyjyxuk3ort2V3a2x574E/Z08S+CE+F99cf2x4gS10N7HUNH1LWGnh0O/a02rc28bPs2jLwlVztDhl6GuOjgqlL2Und2Vmm/hdt1+R9HmPE2Dx7x1O
HJT5qilGcYWdWCndwm0ua70nd2u1aXQwfh98I/Hum/CrVfC154U8aW+oy+Dr3SUOpeLIbrSzctalESO1EpCAtgKcAID2FY0cPXjRdNwlfla1kmr27XP
RzLOcrqZnTx1PEUXFV4T92jKNTlU7tufKr2Wr6s9G+Htp48+GnjrxJF/wr2+1jS9butMkGqW2o2kaQLHYW1vKWR5A52tG54HIHFdlFV6FSS9ndO2t12
SfU+ZzKeV5pgqMvrihOmqnuuM23epOcbNRtqmt3p1KXw78OeMIfhlf/DDxJ8NLptLnXV0bV5tQtJLaQSz3E0J8sSGQZLoBxlTg8YzU0IVVReGqUtPe1
uratteZ0ZnisvlmEM6wmNXOvZe4ozUlyxhGWvLy6Wb31Wmp6z8BvBcnw++DvhDQrqxXT9RttOh+3wqwY/a2UNOzMCQzGQuS2TkmvRwlJ0aEINWaWvr1
/E+P4gx0cxzXEYmEuaEpPlf91O0dOiUbWR31dZ88FABQAUAFABQAUAFABQAUAFABQAUAFABQAUAFABQAUAFABQAUAFABQAUAFABQAUAYus/8fS/7n9T
QBqWn/HrF/uL/ACoAmoAKACgAoATFABigAxQAtABQAUAFABQAUAFABQAUAFABQAUAFABQAUAFABQAUAFABQAUAFABQAUAFABQAUAFABQBi6z/AMfS/w
C5/U0Aalp/x6xf7i/yoAmoAKACgAoAKACgAoAKACgAoAK

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

In this chapter, we covered how to give your `TinyAgent` the ability to process images alongside text. This was only possible because the underlying model, Gemma 4, had native multimodal capabilities!

In [1]:
from illustrated_agents.chapters.ch9 import what_we_built; what_we_built

╭───────────────────────────────────────────────── What We Built ─────────────────────────────────────────────────╮
│ TinyAgent                                                                                                       │
│ ├── agent.py    ← Updated (Allow the agent to process images in addition to text.)                              │
│ ├── llm.py                                                                                                      │
│ ├── memory.py   ← Updated (Track images in the conversation history for the Agent to access.)                   │
│ ├── planning.py                                                                                                 │
│ ├── skills.py                                                                                                   │
│ ├── toolbox.py                                                                                                  │
│ └── tools.py                                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# What's Next

In the next chapter, we are going to be doing a lot! We are going to be building a coding Agent that is capable of creating and running code. It is going to be an interesting exploration as we cover what it means to give an Agent more access to your environment. Security is therefore key ;)